In [243]:
import json
import jsonlines
import openai
import unicodedata
import random
import spacy

from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv()
nlp = spacy.load('en_core_web_sm')

In [223]:
%load_ext blackcellmagic

In [106]:
with open('./data/jokes.json') as f:
    data = json.load(f)

In [190]:
EXCLUDE_IDS = set([88572, 99457, 99483])

jokes = [d['body'] for d in data if d['id'] not in EXCLUDE_IDS]

print('\n\n'.join(jokes[:5]))

Yes, I know what you guys are thinking, "Hey, it's the guy from Twitter."

I'm glad to be on cable. The truth is, I've dreamed of being a talk show host on basic cable ever since I was 46.

I'm happy to report that we're already #1 in TBS's key demographic—people who can't afford HBO.

It's not easy doing a late-night show on a channel without a lot of money and that viewers have trouble finding. So that's why I left NBC.

That's right—the whitest man in show business is back…on the second blackest channel on TV.


In [191]:
counts = defaultdict(int)
jokes_cleaned = []

for joke in jokes:
    if "Conan" in joke:
        continue
        
    joke_clean = (
        unicodedata.normalize("NFKD", joke)
        .replace("  ", " ")
        .replace("—", "--")
        .strip()
    )

    tokens = nlp(joke_clean)
    sentences = [sent.text for sent in tokens.sents]
    sentence_ct = len(sentences)

    entry = {"text": joke_clean, "sentences": sentences, "sentence_ct": sentence_ct}
    jokes_cleaned.append(entry)
    counts[sentence_ct] += 1

print(counts)

defaultdict(<class 'int'>, {1: 844, 2: 8783, 3: 347, 4: 20, 7: 1, 5: 6})


In [192]:
# sample jokes with given num of sentences
sentence_ct = 1
sample_n = 5
filtered = [j["text"] for j in jokes_cleaned if j["sentence_ct"] == sentence_ct]

print("\n\n".join(random.sample(filtered, sample_n)))

Is it weird that my wife will only have sex if I Face Swap with her personal trainer?

My kids asked me what the Wall Street protestors were angry about, & I told them it was the crappy Father’s Day gift they gave me last year.

From now on, rather than a gendered pronoun, I would like to be referred to as the elusive fifth flavor 
umami."

Just called my broker and told him to buy 300 shares of Neil Patrick Harris.

I hope oil stays at under $50 a barrel, because Valentine's Day is coming up.


In [197]:
MAX_ENTRIES = 2000
i = 0

# format and output training data
with jsonlines.open("./data/training.jsonl", mode="w") as f:
    for joke in jokes_cleaned:
        if joke["sentence_ct"] < 2:
            continue
        
        i += 1
        if i > MAX_ENTRIES:
            break

        prompt = "{}\n\n###\n\n".format(joke["sentences"][0])
        completion = " {} END".format(" ".join(joke["sentences"][1:]))

        f.write({"prompt": prompt, "completion": completion})

In [198]:
!openai tools fine_tunes.prepare_data -f "./data/training.jsonl"

Analyzing...

- Your file contains 2000 prompt-completion pairs
- All prompts end with suffix `\n\n###\n\n`
- All completions end with suffix ` END`

No remediations found.

You can use your file for fine-tuning:
> openai api fine_tunes.create -t "./data/training.jsonl"

After you’ve fine-tuned a model, remember that your prompt has to end with the indicator string `\n\n###\n\n` for the model to start generating completions, rather than continuing with the prompt. Make sure to include `stop=[" END"]` so that the generated texts ends at the expected place.
Once your model starts training, it'll approximately take 29.91 minutes to train a `curie` model, and less for `ada` and `babbage`. Queue will approximately take half an hour per job ahead of you.


In [ ]:
!openai api fine_tunes.create \
  -t "./data/training.jsonl" \
  -m davinci \
  --suffix "conan" \
  --n_epochs 2 \
  --learning_rate_multiplier 0.1

In [ ]:
!openai api fine_tunes.list

In [ ]:
!openai api fine_tunes.get -i ft-BU9bjBplVBbc6xamGt3TUlzE

In [244]:
PROMPT_EXAMPLE = '{}\n\n###\n\n'.format(jokes_cleaned[-30]['sentences'][0])
print(PROMPT_EXAMPLE)

openai.Completion.create(
    model=os.environ['OPENAI_MODEL'],
    prompt=PROMPT_EXAMPLE,
    max_tokens=100,
    temperature=0.5,
    n=2,
    stop=[" END"]
)

It's been reported that Donald Trump's family fortune originated from a brothel started by his grandfather.

###




<OpenAIObject text_completion id=cmpl-6Wt1BPNrD3x5xFPlyLzMlovk1esdP at 0x1129f03b0> JSON: {
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "text": " The brothel was called \"Trump's Grandmother's House.\""
    },
    {
      "finish_reason": "stop",
      "index": 1,
      "logprobs": null,
      "text": " Trump's grandfather was inspired by the old Chinese proverb, \"A man's got to make a living somehow.\""
    }
  ],
  "created": 1673296005,
  "id": "cmpl-6Wt1BPNrD3x5xFPlyLzMlovk1esdP",
  "model": "davinci:ft-personal:conan-2023-01-09-20-13-35",
  "object": "text_completion",
  "usage": {
    "completion_tokens": 35,
    "prompt_tokens": 24,
    "total_tokens": 59
  }
}